# ☕ Coffee Price Prediction — Green Coffee Value Benchmark
**Author:** Amit Dwivedi  
**Dataset:** Coffee Value Benchmark (Kaggle)  
**Objective:** Predict `price_usd_per_kg` of green coffee lots using pre-export observable features  
**Date:** 2025

---

## Section 0 — Setup & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
import os
import random

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.pipeline           import Pipeline
from sklearn.compose            import ColumnTransformer
from sklearn.preprocessing      import OneHotEncoder, StandardScaler
from sklearn.impute             import SimpleImputer, KNNImputer
from sklearn.model_selection    import GroupKFold, cross_validate
from sklearn.linear_model       import Ridge
from sklearn.ensemble           import RandomForestRegressor, StackingRegressor
from sklearn.metrics            import mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy              import DummyRegressor
from sklearn.inspection         import PartialDependenceDisplay

# ── Gradient Boosting ─────────────────────────────────────────────────────────
from xgboost  import XGBRegressor
from lightgbm import LGBMRegressor

# ── Interpretability ──────────────────────────────────────────────────────────
import shap

# ── Hyperparameter optimisation ───────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Global settings ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

# ── Data paths ─────────────────────────────────────────────────────────────────
DATA_DIR = 'coffee value benchmark project'
MAIN_CSV  = os.path.join(DATA_DIR, 'coffee_value_benchmark.csv')
FEAT_CSV  = os.path.join(DATA_DIR, 'feature_dictionary.csv')
SPLIT_CSV = os.path.join(DATA_DIR, 'coffee_benchmark_splits.csv')
META_CSV  = os.path.join(DATA_DIR, 'coffee_record_metadata.csv')

print('✅ Environment ready.')

---
## Section 1 — Business Problem Statement

### Context
Green coffee is traded globally before roasting, and its price per kilogram is determined by a complex interplay of:
- **Origin** (country, altitude, climate)
- **Processing** method (Washed, Natural, Honey, Anaerobic)
- **Physical characteristics** (bean size, density, defect rate, moisture)
- **Commercial factors** (certification, buyer segment, lot size, traceability)

### Problem Statement
> *A coffee trading company wants to estimate the market price (USD/kg) of a green coffee lot **before** it reaches market, using only observable pre-export attributes. This enables procurement teams, farmers, and exporters to make data-driven pricing and negotiation decisions.*

### Target Variable
- **`price_usd_per_kg`** — continuous float (regression problem)

### Success Criteria
| Metric | Primary? | Rationale |
|--------|----------|-----------|
| RMSE   | ✅ Yes   | Penalises large price errors; critical for commodity trading |
| MAE    | Secondary | Interpretable in dollar terms |
| R²     | Secondary | Variance explained; baseline comparability |

### Anti-Leakage Rule
The feature dictionary marks `acidity_score`, `sweetness_score`, `body_score`, `aroma_score`, `flavor_score`, `coffee_quality_score`, and `quality_grade` as **Outcome** variables — they are unavailable at prediction time and **must not** be used as features.

---
## Section 2 — Data Loading & Inspection

In [ ]:
# Load all four files
df_main   = pd.read_csv(MAIN_CSV)
df_feat   = pd.read_csv(FEAT_CSV)
df_splits = pd.read_csv(SPLIT_CSV)
df_meta   = pd.read_csv(META_CSV)

print(f'Main dataset   : {df_main.shape}')
print(f'Feature dict   : {df_feat.shape}')
print(f'Splits         : {df_splits.shape}')
print(f'Metadata       : {df_meta.shape}')

In [ ]:
# Attach official train/test split label to the main dataframe
df = df_main.merge(df_splits, on='lot_id', how='left')
df = df.merge(df_meta[['record_id', 'record_type']], on='record_id', how='left')

print('Shape after merge:', df.shape)
print('\nSplit distribution:')
print(df['temporal_split'].value_counts())
print('\nRecord types:')
print(df['record_type'].value_counts())

In [ ]:
# Quick structural overview
df.head(3)

In [ ]:
# Data types and missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
info_df = pd.DataFrame({'dtype': df.dtypes, 'missing_count': missing, 'missing_pct': missing_pct})
info_df[info_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

In [ ]:
# Descriptive statistics for numeric columns
df.describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).T.round(3)

---
## Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Missing value heatmap ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
missing_cols = df.columns[df.isnull().any()].tolist()
sns.heatmap(df[missing_cols].isnull().T, cbar=False, cmap='YlOrRd', ax=ax,
            yticklabels=True, xticklabels=False)
ax.set_title('Missing Value Pattern (yellow = missing)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2 Target variable distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['price_usd_per_kg'].dropna(), bins=50, color='#3b6ea5', edgecolor='white')
axes[0].set_title('Price Distribution (Raw)')
axes[0].set_xlabel('USD per kg')

axes[1].hist(np.log1p(df['price_usd_per_kg'].dropna()), bins=50, color='#e07b39', edgecolor='white')
axes[1].set_title('Price Distribution (log1p transformed)')
axes[1].set_xlabel('log(1 + USD/kg)')

plt.suptitle('Target Variable: price_usd_per_kg', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Skewness (raw)    : {df["price_usd_per_kg"].skew():.3f}')
print(f'Skewness (log1p)  : {np.log1p(df["price_usd_per_kg"]).skew():.3f}')

In [ ]:
# ── 3.3 Price by species ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='species', y='price_usd_per_kg', ax=axes[0],
            palette='Set2', order=df.groupby('species')['price_usd_per_kg'].median().sort_values(ascending=False).index)
axes[0].set_title('Price by Species')

sns.boxplot(data=df, x='intended_buyer_segment', y='price_usd_per_kg', ax=axes[1],
            palette='Set1', order=df.groupby('intended_buyer_segment')['price_usd_per_kg'].median().sort_values(ascending=False).index)
axes[1].set_title('Price by Buyer Segment')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Price by country of origin (top 10 by volume) ────────────────────────
top_countries = df['country_of_origin'].value_counts().head(10).index
df_top = df[df['country_of_origin'].isin(top_countries)]

order = df_top.groupby('country_of_origin')['price_usd_per_kg'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df_top, x='country_of_origin', y='price_usd_per_kg',
            order=order, palette='tab10', ax=ax)
ax.set_title('Price Distribution by Country of Origin (Top 10 Countries)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Price by certification ────────────────────────────────────────────────
order = df.groupby('certification')['price_usd_per_kg'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df, x='certification', y='price_usd_per_kg', order=order,
            palette='pastel', ax=ax)
ax.set_title('Price by Certification')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6 Price vs altitude scatter (coloured by species) ──────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for sp, grp in df.groupby('species'):
    ax.scatter(grp['altitude_mean_meters'], grp['price_usd_per_kg'],
               alpha=0.3, s=15, label=sp)
ax.set_xlabel('Altitude (m)')
ax.set_ylabel('Price (USD/kg)')
ax.set_title('Price vs Altitude by Species')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.7 Correlation heatmap (numeric features) ────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude outcome/target leakage columns
leakage_cols = ['acidity_score','sweetness_score','body_score','aroma_score',
                'flavor_score','coffee_quality_score']
numeric_feat_cols = [c for c in numeric_cols if c not in leakage_cols + ['harvest_calendar_year']]

corr = df[numeric_feat_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=False, linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

# Top correlations with price
print('\nTop correlations with price_usd_per_kg:')
print(corr['price_usd_per_kg'].drop('price_usd_per_kg').abs().sort_values(ascending=False).head(12))

In [ ]:
# ── 3.8 Processing method vs price ───────────────────────────────────────────
order = df.groupby('processing_method')['price_usd_per_kg'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df, x='processing_method', y='price_usd_per_kg',
            order=order, palette='Set3', ax=ax)
ax.set_title('Price by Processing Method')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.9 Outlier detection using IQR ──────────────────────────────────────────
Q1 = df['price_usd_per_kg'].quantile(0.25)
Q3 = df['price_usd_per_kg'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = df[(df['price_usd_per_kg'] < lower) | (df['price_usd_per_kg'] > upper)]
print(f'Price range: [{df["price_usd_per_kg"].min():.2f}, {df["price_usd_per_kg"].max():.2f}]')
print(f'IQR fence  : [{lower:.2f}, {upper:.2f}]')
print(f'Outliers   : {len(outliers):,} ({len(outliers)/len(df)*100:.1f}%)')

---
## Section 4 — Feature Engineering

In [ ]:
# ── 4.1 Identify usable features (no leakage) ─────────────────────────────────
OUTCOME_COLS = ['acidity_score', 'sweetness_score', 'body_score',
                'aroma_score', 'flavor_score', 'coffee_quality_score', 'quality_grade']
META_COLS    = ['record_id', 'lot_id', 'record_type', 'temporal_split']
TARGET_COL   = 'price_usd_per_kg'

FEATURE_COLS = [c for c in df.columns
                if c not in OUTCOME_COLS + META_COLS + [TARGET_COL]]
print(f'Number of features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

In [ ]:
# ── 4.2 Parse harvest_season into numeric year_start ──────────────────────────
df['season_year_start'] = df['harvest_season'].str.extract(r'(\d{4})').astype(float)

# ── 4.3 Log-transform the target ──────────────────────────────────────────────
df['log_price'] = np.log1p(df[TARGET_COL])

# ── 4.4 Interaction features ──────────────────────────────────────────────────
# Species × Altitude (Arabica at high altitude → premium)
df['arabica_flag'] = (df['species'] == 'Arabica').astype(int)
df['arabica_x_altitude'] = df['arabica_flag'] * df['altitude_mean_meters'].fillna(0)

# Lot size tiers (log scale)
df['log_lot_size'] = np.log1p(df['lot_size_kg'])

# Certification flag (any certification vs. None)
df['has_certification'] = (df['certification'] != 'None').astype(int)

# Specialty flag
df['is_specialty'] = (df['intended_buyer_segment'] == 'Specialty').astype(int)

# Premium combo
df['cert_x_specialty'] = df['has_certification'] * df['is_specialty']

print('New engineered features added.')
print(df[['harvest_season','season_year_start','arabica_x_altitude',
          'log_lot_size','has_certification','is_specialty','log_price']].head(3))

In [ ]:
# ── 4.5 Define final feature lists ────────────────────────────────────────────
# Remove original columns that were replaced or are IDs
DROP_COLS = ['harvest_season', 'lot_size_kg', 'arabica_flag']

FEATURE_COLS_ENG = [c for c in df.columns
                    if c not in OUTCOME_COLS + META_COLS + [TARGET_COL, 'log_price']
                    and c not in DROP_COLS]

# Categorical and numeric splits
CAT_COLS = df[FEATURE_COLS_ENG].select_dtypes(include=['object', 'category']).columns.tolist()
NUM_COLS = df[FEATURE_COLS_ENG].select_dtypes(include=[np.number]).columns.tolist()

print(f'Total features : {len(FEATURE_COLS_ENG)}')
print(f'Categorical    : {CAT_COLS}')
print(f'\nNumeric        : {NUM_COLS}')

---
## Section 5 — Train / Test Split

In [ ]:
# ── Use the official temporal split from coffee_benchmark_splits.csv ──────────
df_model = df.dropna(subset=[TARGET_COL]).copy()

train_mask = df_model['temporal_split'] == 'Train'
test_mask  = df_model['temporal_split'] == 'Test (Temporal)'

X_train = df_model.loc[train_mask, FEATURE_COLS_ENG].reset_index(drop=True)
y_train = df_model.loc[train_mask, 'log_price'].reset_index(drop=True)
groups_train = df_model.loc[train_mask, 'lot_id'].reset_index(drop=True)

X_test  = df_model.loc[test_mask, FEATURE_COLS_ENG].reset_index(drop=True)
y_test  = df_model.loc[test_mask, 'log_price'].reset_index(drop=True)
y_test_raw  = df_model.loc[test_mask, TARGET_COL].reset_index(drop=True)

print(f'Train size : {X_train.shape[0]:,} rows | {X_train.shape[1]} features')
print(f'Test  size : {X_test.shape[0]:,} rows')

---
## Section 6 — Preprocessing Pipeline & Model Training

In [ ]:
# ── 6.1 Build preprocessing pipeline ─────────────────────────────────────────
numeric_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     NUM_COLS),
    ('cat', categorical_transformer, CAT_COLS)
], remainder='drop')

print('Preprocessor defined.')

In [ ]:
# ── 6.2 Cross-validation helper (GroupKFold on lot_id) ───────────────────────
cv = GroupKFold(n_splits=5)

def evaluate_cv(model, name):
    """Run GroupKFold CV and return RMSE, MAE, R2 on log-price."""
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    scores = cross_validate(
        pipe, X_train, y_train, groups=groups_train, cv=cv,
        scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
        n_jobs=-1, return_train_score=False
    )
    rmse = -scores['test_neg_root_mean_squared_error'].mean()
    mae  = -scores['test_neg_mean_absolute_error'].mean()
    r2   =  scores['test_r2'].mean()
    print(f'[{name:25s}] CV RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')
    return {'name': name, 'cv_rmse': rmse, 'cv_mae': mae, 'cv_r2': r2}

results = []

In [ ]:
# ── 6.3 Baseline: Median Predictor ────────────────────────────────────────────
baseline = DummyRegressor(strategy='median')
results.append(evaluate_cv(baseline, 'Median Baseline'))

In [ ]:
# ── 6.4 Ridge Regression ──────────────────────────────────────────────────────
# Ridge uses a closed-form solver; random_state is not a valid parameter
ridge = Ridge(alpha=1.0)
results.append(evaluate_cv(ridge, 'Ridge Regression'))

In [ ]:
# ── 6.5 Random Forest ─────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=12,
                           min_samples_leaf=5, random_state=SEED, n_jobs=-1)
results.append(evaluate_cv(rf, 'Random Forest'))

In [ ]:
# ── 6.6 XGBoost ───────────────────────────────────────────────────────────────
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8,
                   tree_method='hist', random_state=SEED, verbosity=0)
results.append(evaluate_cv(xgb, 'XGBoost'))

In [ ]:
# ── 6.7 LightGBM ──────────────────────────────────────────────────────────────
lgbm = LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8,
                     random_state=SEED, verbose=-1)
results.append(evaluate_cv(lgbm, 'LightGBM'))

---
## Section 7 — Hyperparameter Tuning with Optuna

In [ ]:
# ── 7.1 Tune XGBoost ──────────────────────────────────────────────────────────
def xgb_objective(trial):
    # Note: tree_method, random_state, verbosity are fixed constants — not tuned
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 600),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 9),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = XGBRegressor(**params)
    pipe  = Pipeline([('pre', preprocessor), ('model', model)])
    scores = cross_validate(
        pipe, X_train, y_train, groups=groups_train, cv=cv,
        scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    return -scores['test_neg_root_mean_squared_error'].mean()

study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=40, show_progress_bar=True)

print('\nBest XGBoost params:')
print(study_xgb.best_params)
print(f'Best CV RMSE: {study_xgb.best_value:.4f}')

In [ ]:
# ── 7.2 Tune LightGBM ─────────────────────────────────────────────────────────
def lgbm_objective(trial):
    # Note: random_state and verbose are fixed constants — not tuned
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 600),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 3, 9),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 150),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
    }
    model = LGBMRegressor(**params)
    pipe  = Pipeline([('pre', preprocessor), ('model', model)])
    scores = cross_validate(
        pipe, X_train, y_train, groups=groups_train, cv=cv,
        scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    return -scores['test_neg_root_mean_squared_error'].mean()

study_lgbm = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgbm.optimize(lgbm_objective, n_trials=40, show_progress_bar=True)

print('\nBest LightGBM params:')
print(study_lgbm.best_params)
print(f'Best CV RMSE: {study_lgbm.best_value:.4f}')

In [ ]:
# ── 7.3 Build tuned models ────────────────────────────────────────────────────
# best_params only contains the trial's sampled keys (n_estimators, learning_rate, etc.)
# Fixed constants (tree_method, random_state, verbosity) are added here without collision.
xgb_tuned  = XGBRegressor(**study_xgb.best_params,
                           tree_method='hist', random_state=SEED, verbosity=0)
lgbm_tuned = LGBMRegressor(**study_lgbm.best_params,
                           random_state=SEED, verbose=-1)

results.append(evaluate_cv(xgb_tuned,  'XGBoost (Tuned)'))
results.append(evaluate_cv(lgbm_tuned, 'LightGBM (Tuned)'))

In [ ]:
# ── 7.4 Stacking Ensemble ─────────────────────────────────────────────────────
# StackingRegressor does not accept n_jobs directly; parallelism is handled internally
estimators = [
    ('xgb',  xgb_tuned),
    ('lgbm', lgbm_tuned),
    ('rf',   RandomForestRegressor(n_estimators=200, max_depth=12,
                                   min_samples_leaf=5, random_state=SEED, n_jobs=-1))
]
stacker = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5
)
results.append(evaluate_cv(stacker, 'Stacking Ensemble'))

---
## Section 8 — Evaluation on Test Set

In [ ]:
# ── 8.1 Train best models on full training set and evaluate on test set ───────
def test_eval(model, name, X_tr, y_tr_log, X_te, y_te_raw):
    """Fit pipeline, predict on test, report metrics in original price scale."""
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    pipe.fit(X_tr, y_tr_log)
    preds_log = pipe.predict(X_te)
    preds_raw = np.expm1(preds_log)   # invert log1p
    preds_raw = np.clip(preds_raw, 0, None)

    rmse = np.sqrt(mean_squared_error(y_te_raw, preds_raw))
    mae  = mean_absolute_error(y_te_raw, preds_raw)
    r2   = r2_score(y_te_raw, preds_raw)
    print(f'[{name:25s}] Test RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')
    return pipe, preds_raw, {'name': name, 'test_rmse': rmse, 'test_mae': mae, 'test_r2': r2}

test_results = {}
models_to_test = [
    ('Median Baseline',   DummyRegressor(strategy='median')),
    ('Ridge Regression',  Ridge(alpha=1.0)),
    ('Random Forest',     RandomForestRegressor(n_estimators=200, max_depth=12,
                                                min_samples_leaf=5, random_state=SEED, n_jobs=-1)),
    ('XGBoost (Tuned)',   xgb_tuned),
    ('LightGBM (Tuned)',  lgbm_tuned),
    ('Stacking Ensemble', stacker),
]

fitted_pipes = {}
for mname, mobj in models_to_test:
    pipe, preds, tres = test_eval(mobj, mname, X_train, y_train, X_test, y_test_raw)
    test_results[mname] = tres
    fitted_pipes[mname] = (pipe, preds)

In [ ]:
# ── 8.2 Summary metrics table ─────────────────────────────────────────────────
cv_df   = pd.DataFrame(results).set_index('name')
test_df = pd.DataFrame(test_results.values()).set_index('name')
summary = cv_df.join(test_df, how='outer').round(4)
summary.columns = ['CV RMSE (log)', 'CV MAE (log)', 'CV R²',
                   'Test RMSE (USD)', 'Test MAE (USD)', 'Test R²']
print('\n═══════════════════ MODEL COMPARISON TABLE ═══════════════════')
print(summary.to_string())

In [ ]:
# ── 8.3 Bar chart: Test RMSE comparison ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
test_rmse_series = summary['Test RMSE (USD)'].dropna().sort_values()
colors = ['#d4380d' if v == test_rmse_series.min() else '#8c8c8c' for v in test_rmse_series.values]
bars = ax.barh(test_rmse_series.index, test_rmse_series.values, color=colors, edgecolor='white')
ax.set_xlabel('RMSE (USD/kg)')
ax.set_title('Test Set RMSE by Model (lower is better)', fontsize=12)
for bar, val in zip(bars, test_rmse_series.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.4 Residual analysis for best model ─────────────────────────────────────
best_model_name = summary['Test RMSE (USD)'].dropna().idxmin()
best_pipe, best_preds = fitted_pipes[best_model_name]
residuals = y_test_raw.values - best_preds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(best_preds, residuals, alpha=0.3, s=10, color='#3b82d4')
axes[0].axhline(0, color='red', linewidth=1.2)
axes[0].set_xlabel('Predicted Price (USD/kg)')
axes[0].set_ylabel('Residual (Actual − Predicted)')
axes[0].set_title(f'Residuals vs Predicted — {best_model_name}')

axes[1].hist(residuals, bins=50, color='#7c5cd8', edgecolor='white')
axes[1].axvline(0, color='red', linewidth=1.2)
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# ── 8.5 Actual vs Predicted scatter ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test_raw, best_preds, alpha=0.3, s=12, color='#3b82d4')
max_val = max(y_test_raw.max(), best_preds.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Price (USD/kg)')
ax.set_ylabel('Predicted Price (USD/kg)')
ax.set_title(f'Actual vs Predicted — {best_model_name}')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 9 — Model Interpretability

In [ ]:
# ── 9.1 Prepare SHAP-compatible transformed data ──────────────────────────────
# Use the LightGBM tuned model (best individual model for SHAP efficiency)
lgbm_pipe = Pipeline([('pre', preprocessor), ('model', lgbm_tuned)])
lgbm_pipe.fit(X_train, y_train)

# Transform the test set
X_test_transformed = lgbm_pipe['pre'].transform(X_test)

# Get feature names after encoding
ohe_names = lgbm_pipe['pre'].named_transformers_['cat']['ohe'].get_feature_names_out(CAT_COLS).tolist()
feature_names_transformed = NUM_COLS + ohe_names

print(f'Transformed feature count: {X_test_transformed.shape[1]}')

In [ ]:
# ── 9.2 SHAP values ───────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(lgbm_pipe['model'])
shap_values = explainer.shap_values(X_test_transformed)
# LightGBM TreeExplainer may return a list (one array per class/output).
# For a single-output regressor it returns a 2-D array; normalise to 2-D ndarray.
if isinstance(shap_values, list):
    shap_values = shap_values[0]

# lgbm test-set predictions (used as index source for waterfall plots)
lgbm_preds_test = np.expm1(np.clip(lgbm_pipe.predict(X_test), 0, None))

# Summary bar plot (global importance)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_transformed,
                  feature_names=feature_names_transformed,
                  plot_type='bar', max_display=20, show=False)
plt.title('SHAP Feature Importance (LightGBM — Top 20)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3 SHAP beeswarm plot ────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_transformed,
                  feature_names=feature_names_transformed,
                  max_display=20, show=False)
plt.title('SHAP Beeswarm (Direction + Magnitude)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.4 SHAP waterfall — highest predicted price lot (lgbm) ─────────────────
# Use lgbm_preds_test so the index aligns with the SHAP values from lgbm_pipe
high_idx = int(np.argmax(lgbm_preds_test))
explanation = shap.Explanation(
    values=shap_values[high_idx],
    base_values=float(explainer.expected_value),
    data=X_test_transformed[high_idx],
    feature_names=feature_names_transformed
)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(explanation, max_display=15, show=False)
plt.title('SHAP Waterfall — Highest Predicted Price Lot', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.5 SHAP waterfall — lowest predicted price lot (lgbm) ──────────────────
low_idx = int(np.argmin(lgbm_preds_test))
explanation_low = shap.Explanation(
    values=shap_values[low_idx],
    base_values=float(explainer.expected_value),
    data=X_test_transformed[low_idx],
    feature_names=feature_names_transformed
)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(explanation_low, max_display=15, show=False)
plt.title('SHAP Waterfall — Lowest Predicted Price Lot', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.6 Partial Dependence Plots ──────────────────────────────────────────────
# Use Random Forest (simpler, scikit-learn native PDP)
rf_pipe = Pipeline([('pre', preprocessor), ('model',
    RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=5,
                          random_state=SEED, n_jobs=-1))])
rf_pipe.fit(X_train, y_train)

# Find indices of key features in the transformed array
pdp_features_original = ['altitude_mean_meters', 'defect_rate_pct', 'log_lot_size']
pdp_indices = [NUM_COLS.index(f) for f in pdp_features_original if f in NUM_COLS]

fig, axes = plt.subplots(1, len(pdp_indices), figsize=(5 * len(pdp_indices), 4))
PartialDependenceDisplay.from_estimator(
    rf_pipe, X_test, features=pdp_indices,
    feature_names=FEATURE_COLS_ENG,
    ax=axes if len(pdp_indices) > 1 else [axes],
    kind='average', grid_resolution=50
)
plt.suptitle('Partial Dependence Plots (Random Forest)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 10 — Conclusion & Business Insights

In [ ]:
# ── 10.1 Final summary ────────────────────────────────────────────────────────
print('═' * 60)
print('   FINAL MODEL PERFORMANCE SUMMARY')
print('═' * 60)
print(summary[['Test RMSE (USD)', 'Test MAE (USD)', 'Test R²']].to_string())
print('═' * 60)
print(f'\n✅ Best Model : {best_model_name}')
br = summary.loc[best_model_name]
print(f'   Test RMSE  : ${br["Test RMSE (USD)"]:.3f} USD/kg')
print(f'   Test MAE   : ${br["Test MAE (USD)"]:.3f} USD/kg')
print(f'   Test R²    : {br["Test R²"]:.4f}')

### Key Business Insights

1. **Species is the strongest price driver** — Arabica commands a consistently higher price than Robusta across all origins. This validates the fundamental commodity market segmentation.

2. **Buyer segment stratification** — Specialty-targeted lots command 2–3× the price of commodity-grade lots. Pre-classifying a lot's commercial intent is a key pricing signal.

3. **Altitude premium (Arabica-specific)** — For Arabica, altitude above 1,500m correlates strongly with higher prices. This reflects the "high-grown" quality premium well-established in coffee trade.

4. **Certifications add measurable value** — Organic and Fair Trade certified lots fetch higher prices, particularly when targeting the Specialty segment. The `cert_x_specialty` interaction feature captures this compound premium.

5. **Processing method matters** — Anaerobic and Natural processed coffees tend toward higher prices, reflecting premium labour-intensive post-harvest techniques.

6. **Defect rate is a value destroyer** — Higher defect rates consistently suppress prices, confirming that green coffee quality control is commercially rewarded.

### Limitations

- The dataset is **synthetic** (benchmark) — real-world price volatility from commodity markets, FX rates, and geopolitical factors is not captured.
- The temporal split may not fully reflect real market cycles if the synthetic data does not encode true time-series dynamics.
- `country_of_origin` is encoded simply here; a more robust approach would use target encoding with regularization.

### Future Work

- Integrate **real commodity price indices** (ICE Futures) as market-level features.
- Apply **neural network tabular models** (TabNet, FT-Transformer) for comparison.
- Build a **REST API** with FastAPI to serve real-time price predictions.
- Add **confidence intervals** via quantile regression for procurement risk management.

In [ ]:
print('Project complete. All deliverables generated.')
print('Author: Amit Dwivedi')